# 02 — Training both models

Both models are trained by the same loop (`pertreadout.train.train_model`), on the same dataloaders,
under the same loss — GEARS' autofocus MSE plus its direction-aware term — with the same optimiser
and schedule. The only thing that differs is which module was handed in.

That matters more than it sounds. Most reported gaps between perturbation models are confounded by
training setup; this one cannot be, because there is only one training setup.

To reproduce everything from a clean checkout:

```bash
pip install -e ".[full,dev]"
python -m pertreadout.cli download
python -m pertreadout.cli manifest
python -m pertreadout.cli reproduce --config configs/norman.yaml
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

RUNS = REPO / "runs"
TABLES = REPO / "results" / "tables"
pd.set_option("display.width", 160)

## What each run produced

A run does not save a score. It saves `predictions.npz`: the condition-mean predicted and true
profiles on the held-out perturbations, the control mean, the per-condition DE gene indices, and the
standard error of each measured delta. Every readout in `03` and `04` is computed from those files
afterwards, so the same predictions can be scored every way the literature scores them.

In [ ]:
import json

from pertreadout import evaluate

runs = evaluate.find_runs(RUNS)
meta = []
for row in runs.itertuples():
    info = json.loads((row.path.parent / "run.json").read_text())
    meta.append({k: info[k] for k in
                 ["dataset", "model", "seed", "use_priors", "n_held_out_conditions",
                  "n_parameters", "seconds"]} | {"final_train_loss": info["train_loss"][-1]})
meta = pd.DataFrame(meta)
print(f"{len(meta)} runs, {meta['seconds'].sum()/3600:.1f} CPU-hours total")
meta.groupby(["dataset", "model", "use_priors"]).agg(
    seeds=("seed", "nunique"), held_out=("n_held_out_conditions", "first"),
    params=("n_parameters", "first"), final_loss=("final_train_loss", "mean"),
    sec_per_run=("seconds", "mean")).round(3)

## Loss curves

Six epochs with `StepLR(step_size=1, gamma=0.5)`, which is GEARS' own schedule — its learning rate is
already down by 32x at epoch 6. These runs are truncated versions of that schedule, not a different
one, but they are truncated: see the limitations section of the README.

In [ ]:
fig, axes = plt.subplots(1, runs["dataset"].nunique(), figsize=(4.2 * runs["dataset"].nunique(), 3.6),
                         squeeze=False)
for ax, dataset in zip(axes[0], sorted(runs["dataset"].unique())):
    for model, colour in [("gears", "#3B6EA5"), ("hgnn", "#C1553B")]:
        curves = [json.loads((r.path.parent / "run.json").read_text())["train_loss"]
                  for r in runs.itertuples() if r.dataset == dataset and r.model == model]
        if not curves:
            continue
        arr = np.array(curves)
        epochs = np.arange(1, arr.shape[1] + 1)
        ax.plot(epochs, arr.mean(0), color=colour, label=model)
        ax.fill_between(epochs, arr.min(0), arr.max(0), color=colour, alpha=0.2)
    ax.set_title(dataset.capitalize()); ax.set_xlabel("epoch"); ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=8)
axes[0][0].set_ylabel("training loss (mean, min-max over seeds)")
fig.tight_layout(); plt.show()

## A sanity baseline before any comparison

Predict the control mean profile for every held-out condition — a model with no perturbation input at
all. If a trained model does not beat that on genome-wide MSE, its reported readouts are not
measuring perturbation prediction.

This is the check that motivates the whole repository: the same trivial predictor scores >0.95 on
Pearson of absolute expression.

In [ ]:
from pertreadout.readouts import mse_genome_wide, pearson_absolute, pearson_delta
from pertreadout.train import load_predictions

rows = []
for row in runs[runs["model"].isin(["gears", "hgnn"])].itertuples():
    d = load_predictions(row.path)
    trivial = np.tile(d["ctrl_mean"], (d["true"].shape[0], 1))
    rows.append({
        "dataset": row.dataset, "model": row.model, "seed": row.seed,
        "model_mse": mse_genome_wide(d["pred"], d["true"]),
        "control_mean_mse": mse_genome_wide(trivial, d["true"]),
        "model_pearson_abs": pearson_absolute(d["pred"], d["true"]),
        "control_mean_pearson_abs": pearson_absolute(trivial, d["true"]),
        "model_pearson_delta": pearson_delta(d["pred"], d["true"], d["ctrl_mean"]),
    })
baseline = pd.DataFrame(rows)
baseline.groupby(["dataset", "model"]).mean(numeric_only=True).drop(columns="seed").round(4)

Read the two Pearson-absolute columns side by side. Whatever the trained models score there, the
do-nothing predictor is close behind — which is the reason `03` reports Pearson on the delta and
draws that baseline as a line on the absolute panel.